In [6]:
import logging
from datetime import datetime

# Configure production-level logging
def setup_logging():
    """Configure logging for preprocessing pipeline."""
    logger = logging.getLogger('preprocessing')

    if logger.handlers:
        return logger

    logger.setLevel(logging.INFO)

    # Console handler
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

    return logger

logger = setup_logging()
logger.info("Logging configured for preprocessing pipeline")

2026-04-18 15:47:36 - preprocessing - INFO - Logging configured for preprocessing pipeline
INFO:preprocessing:Logging configured for preprocessing pipeline


In [3]:
!pip install numpy pandas scikit-learn imagehash

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 3.5 MB/s eta 0:00:00


In [4]:
!pip install -q opencv-python Pillow pillow-heif numpy pandas scikit-learn imagehash

In [2]:
import cv2
import numpy as np
import pandas as pd
import sklearn
import imagehash

print(np.__version__)  # should show 1.26.x
print("All good ✓")

1.26.4
All good ✓


In [6]:
!pip install -q "numpy<2" opencv-python --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 8.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires nump

In [3]:
from google.colab import drive
from pathlib import Path
import os

# Step 1: Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')
print("✓ Google Drive mounted at /content/drive")

# Step 2: Define paths for YOUR setup
# ===== CUSTOMIZED FOR YOUR DRIVE STRUCTURE =====
gdrive_root = Path('/content/drive/MyDrive')
dataset_source = gdrive_root / 'Mini_Project_Dataset'  # Dataset from "Shared with me"
output_root = gdrive_root / 'Banana_Leaf_Processed'    # New output folder in My Drive

print(f"\nPath Configuration:")
print(f"  Google Drive root: {gdrive_root}")
print(f"  Dataset source: {dataset_source} (from Shared with me)")
print(f"  Output directory: {output_root} (NEW folder in My Drive)")

# Step 3: Verify dataset exists
if dataset_source.exists():
    print(f"\n✓ Dataset found at {dataset_source}")
    diseases = [d.name for d in dataset_source.iterdir() if d.is_dir()]
    print(f"Disease classes found: {sorted(diseases)}")

    # Count total images
    total_images = 0
    for disease_folder in dataset_source.iterdir():
        if disease_folder.is_dir():
            for contributor_folder in disease_folder.iterdir():
                if contributor_folder.is_dir():
                    image_count = len(list(contributor_folder.glob('*')))
                    total_images += image_count
    print(f"Total images in dataset: {total_images}")
else:
    print(f"\n✗ Dataset NOT found at {dataset_source}")
    print("Please ensure Mini_Project_Dataset is accessible in Google Drive")
    print("\nExpected structure:")
    print("  Shared with me/")
    print("    └── Mini_Project_Dataset/")
    print("        ├── healthy_leaves/")
    print("        ├── panama_wilt/")
    print("        ├── potassium_deficiency/")
    print("        └── sigatoka/")

# Step 4: Create output directory structure
processed_dir = output_root / 'processed'
split_dir = output_root / 'split'
metadata_dir = output_root / 'metadata'

for folder in [output_root, processed_dir, split_dir, metadata_dir]:
    folder.mkdir(parents=True, exist_ok=True)
    print(f"✓ Directory created: {folder.name}")

print(f"\n✓ All output directories ready at: {output_root}")

# Update the dataset_root variable used later
dataset_root = dataset_source

Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted at /content/drive

Path Configuration:
  Google Drive root: /content/drive/MyDrive
  Dataset source: /content/drive/MyDrive/Mini_Project_Dataset (from Shared with me)
  Output directory: /content/drive/MyDrive/Banana_Leaf_Processed (NEW folder in My Drive)

✓ Dataset found at /content/drive/MyDrive/Mini_Project_Dataset
Disease classes found: ['healthy_leaves', 'panama_wilt', 'potassium_deficiency', 'sigatoka']
Total images in dataset: 1215
✓ Directory created: Banana_Leaf_Processed
✓ Directory created: processed
✓ Directory created: split
✓ Directory created: metadata

✓ All output directories ready at: /content/drive/MyDrive/Banana_Leaf_Processed


In [4]:
# Create output directory structure
print("Creating output directory structure...")

# Use paths already defined from Google Drive mount
processed_dir = output_root / 'processed'
split_dir = output_root / 'split'
metadata_dir = output_root / 'metadata'

for directory in [processed_dir, split_dir, metadata_dir]:
    directory.mkdir(parents=True, exist_ok=True)
    print(f"  ✓ Created: {directory}")

# Create disease-specific folders
print("\nCreating disease-specific subdirectories...")
for disease in ['healthy_leaves', 'panama_wilt', 'potassium_deficiency', 'sigatoka']:
    (processed_dir / disease).mkdir(exist_ok=True)
    (split_dir / 'train' / disease).mkdir(parents=True, exist_ok=True)
    (split_dir / 'val' / disease).mkdir(parents=True, exist_ok=True)
    (split_dir / 'test' / disease).mkdir(parents=True, exist_ok=True)
    print(f"  ✓ Created folders for: {disease}")

print(f"\n✓ Output directory structure complete")
print(f"Location: {output_root}")

Creating output directory structure...
  ✓ Created: /content/drive/MyDrive/Banana_Leaf_Processed/processed
  ✓ Created: /content/drive/MyDrive/Banana_Leaf_Processed/split
  ✓ Created: /content/drive/MyDrive/Banana_Leaf_Processed/metadata

Creating disease-specific subdirectories...
  ✓ Created folders for: healthy_leaves
  ✓ Created folders for: panama_wilt
  ✓ Created folders for: potassium_deficiency
  ✓ Created folders for: sigatoka

✓ Output directory structure complete
Location: /content/drive/MyDrive/Banana_Leaf_Processed


In [7]:
# Standard imports
import os
import cv2
import numpy as np
import pandas as pd
import imagehash
from PIL import Image, ImageOps
from pathlib import Path
from collections import defaultdict
from typing import Dict, List, Tuple
import json

# Machine learning imports
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
import warnings
warnings.filterwarnings('ignore')

# Register HEIF support for PIL
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
    logger.info("HEIC support enabled")
except ImportError:
    logger.warning("pillow-heif not available - HEIC support limited")

print("✓ All libraries imported successfully")

2026-04-18 15:47:41 - preprocessing - INFO - HEIC support enabled
INFO:preprocessing:HEIC support enabled


✓ All libraries imported successfully


In [8]:
class Config:
    """Production configuration for preprocessing pipeline."""

    # ===== ENVIRONMENT =====
    # This notebook is designed for Google Colab
    # Paths automatically handle Google Drive mounting
    IS_COLAB = True  # Automatically detected via Google Drive

    # Image processing
    TARGET_IMAGE_SIZE = (256, 256)  # (height, width) - CNN standard
    TARGET_COLOR_SPACE = "RGB"
    TARGET_IMAGE_FORMAT = "JPEG"
    JPEG_QUALITY = 90  # 1-100, higher = better quality

    # Supported formats
    SUPPORTED_FORMATS = {".jpg", ".jpeg", ".png", ".heic"}

    # ===== QUALITY THRESHOLDS =====
    # Customize these if needed

    BLUR_THRESHOLD = 100.0           # Laplacian variance
                                      # Higher = stricter blur detection
                                      # Typical range: 80-150

    MIN_BRIGHTNESS = 40              # Too dark threshold (0-255)
                                      # Images darker than this rejected

    MAX_BRIGHTNESS = 240             # Overexposed threshold
                                      # Images brighter than this rejected

    MIN_CONTRAST = 15.0              # Flat image threshold
                                      # Std dev of pixel values

    # ===== DUPLICATE DETECTION =====
    HASH_SIZE = 8                    # 8x8 = 64-bit hash
    HASH_ALGORITHM = "dhash"         # Options: ahash, dhash, phash, whash
    DUPLICATE_THRESHOLD = 0.90       # 90% similarity = duplicate
                                      # Range: 0.80-0.95

    # ===== DATASET SPLITTING =====
    TRAIN_RATIO = 0.70              # 70% for training
    VAL_RATIO = 0.15                # 15% for validation (hyperparameter tuning)
    TEST_RATIO = 0.15               # 15% for testing (final evaluation)
    RANDOM_SEED = 42                # For reproducible splits

    # ===== DEVICE MAPPING =====
    # Maps contributor folder names to device types
    DEVICE_MAPPING = {
        "Vedant_Primary": "iPhone",
        "Vedant_Secondary": "iPhone",
        "Sagar": "Android",
        "Subodh": "Android",
        "Sudhanshu": "Android"
    }

    # ===== DISEASE CLASSES =====
    # Must match your Google Drive folder names exactly
    DISEASE_CLASSES = [
        "healthy_leaves",
        "panama_wilt",
        "potassium_deficiency",
        "sigatoka"
    ]

# Validate configuration
assert Config.TRAIN_RATIO + Config.VAL_RATIO + Config.TEST_RATIO == 1.0
assert Config.DUPLICATE_THRESHOLD >= 0 and Config.DUPLICATE_THRESHOLD <= 1.0

print(f"✓ Configuration loaded:")
print(f"  - Image size: {Config.TARGET_IMAGE_SIZE}")
print(f"  - JPEG quality: {Config.JPEG_QUALITY}")
print(f"  - Split ratio: Train {Config.TRAIN_RATIO*100:.0f}% | Val {Config.VAL_RATIO*100:.0f}% | Test {Config.TEST_RATIO*100:.0f}%")
print(f"  - Quality thresholds:")
print(f"    • Blur: {Config.BLUR_THRESHOLD}")
print(f"    • Brightness: {Config.MIN_BRIGHTNESS}-{Config.MAX_BRIGHTNESS}")
print(f"    • Contrast: {Config.MIN_CONTRAST}")
print(f"  - Duplicate threshold: {Config.DUPLICATE_THRESHOLD*100:.0f}% similarity")
print(f"\nℹ️  To customize: Edit the Config class above and restart")

✓ Configuration loaded:
  - Image size: (256, 256)
  - JPEG quality: 90
  - Split ratio: Train 70% | Val 15% | Test 15%
  - Quality thresholds:
    • Blur: 100.0
    • Brightness: 40-240
    • Contrast: 15.0
  - Duplicate threshold: 90% similarity

ℹ️  To customize: Edit the Config class above and restart


In [9]:
class QualityChecker:
    """Detect problematic images using multiple quality metrics."""

    @staticmethod
    def detect_blur(gray_image):
        """
        Detect blur using Laplacian variance.
        WHY: Blurry images don't show disease patterns clearly.
        Higher variance = sharper image.
        """
        laplacian = cv2.Laplacian(gray_image, cv2.CV_64F)
        return laplacian.var()

    @staticmethod
    def analyze_brightness(gray_image):
        """Detect underexposed/overexposed images."""
        return float(cv2.mean(gray_image)[0])

    @staticmethod
    def analyze_contrast(gray_image):
        """Detect low-contrast (flat) images."""
        return float(gray_image.std())

    @classmethod
    def check_image_quality(cls, image, config=Config):
        """Run complete quality check on image."""
        if image is None or image.size == 0:
            return None

        if len(image.shape) == 3:
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        else:
            gray = image

        blur_score = cls.detect_blur(gray)
        brightness = cls.analyze_brightness(gray)
        contrast = cls.analyze_contrast(gray)

        # Determine if quality is acceptable
        is_blurry = blur_score < config.BLUR_THRESHOLD
        brightness_ok = config.MIN_BRIGHTNESS <= brightness <= config.MAX_BRIGHTNESS
        contrast_ok = contrast >= config.MIN_CONTRAST
        is_quality = not (is_blurry or not brightness_ok or not contrast_ok)

        flags = []
        if is_blurry:
            flags.append(f"BLURRY({blur_score:.0f})")
        if not brightness_ok:
            flags.append(f"BRIGHTNESS({brightness:.0f})")
        if not contrast_ok:
            flags.append(f"CONTRAST({contrast:.0f})")

        return {
            "is_quality": is_quality,
            "blur_score": blur_score,
            "brightness": brightness,
            "contrast": contrast,
            "flags": ";".join(flags) if flags else ""
        }

print("✓ Quality Checker loaded")

✓ Quality Checker loaded


In [10]:
class DuplicateDetector:
    """Detect duplicate/near-duplicate images using perceptual hashing."""

    def __init__(self, hash_size=8, algorithm="dhash", threshold=0.90):
        """
        WHY: Perceptual hashing finds similar images, not identical files.
        This prevents data leakage from same leaf photographed multiple ways.
        """
        self.hash_size = hash_size
        self.algorithm = algorithm
        self.threshold = threshold
        self.image_hashes = {}

    def compute_hash(self, image_path):
        """Compute perceptual hash of image."""
        try:
            if self.algorithm == "dhash":
                return imagehash.dhash(Image.open(image_path), hash_size=self.hash_size)
            elif self.algorithm == "phash":
                return imagehash.phash(Image.open(image_path), hash_size=self.hash_size)
            else:
                return imagehash.dhash(Image.open(image_path), hash_size=self.hash_size)
        except Exception as e:
            logger.warning(f"Could not hash {image_path}: {e}")
            return None

    def compute_similarity(self, hash1, hash2):
        """Compute similarity between two hashes (0.0-1.0)."""
        if hash1 is None or hash2 is None:
            return 0.0
        distance = hash1 - hash2  # Hamming distance
        max_distance = 64  # For 8x8 hash
        similarity = 1.0 - (distance / max_distance)
        return max(0.0, min(1.0, similarity))

    def find_duplicates_in_batch(self, image_paths):
        """Find all duplicate groups in batch of images."""
        hashes = {}
        for path in image_paths:
            hashes[str(path)] = self.compute_hash(path)

        duplicate_groups = []
        processed = set()

        for i, path1 in enumerate(image_paths):
            if str(path1) in processed:
                continue
            group = [path1]

            for path2 in image_paths[i+1:]:
                str_path2 = str(path2)
                if str_path2 in processed:
                    continue

                similarity = self.compute_similarity(hashes[str(path1)], hashes[str_path2])
                if similarity >= self.threshold:
                    group.append(path2)
                    processed.add(str_path2)

            if len(group) > 1:
                duplicate_groups.append(group)
                processed.add(str(path1))

        return {
            "duplicate_groups": duplicate_groups,
            "duplicate_count": sum(len(g) - 1 for g in duplicate_groups),
            "unique_count": len(image_paths) - sum(len(g) - 1 for g in duplicate_groups)
        }

print("✓ Duplicate Detector loaded")

✓ Duplicate Detector loaded


In [11]:
class ImageProcessor:
    """Handle image format conversion, EXIF correction, resizing, normalization."""

    def __init__(self, target_size, jpeg_quality):
        self.target_size = target_size
        self.jpeg_quality = jpeg_quality

    def correct_exif_orientation(self, image_path):
        """Correct rotation using EXIF metadata."""
        try:
            img = Image.open(image_path)
            img = ImageOps.exif_transpose(img)
            return img if img is not None else Image.open(image_path)
        except:
            return Image.open(image_path)

    def resize_with_padding(self, image, target_size):
        """Resize image preserving aspect ratio with padding."""
        h, w = image.shape[:2]
        target_h, target_w = target_size

        scale = min(target_w / w, target_h / h)
        new_w, new_h = int(w * scale), int(h * scale)

        resized = cv2.resize(image, (new_w, new_h))

        # Create white canvas
        if len(image.shape) == 3:
            canvas = np.full((target_h, target_w, image.shape[2]), 255, dtype=np.uint8)
        else:
            canvas = np.full((target_h, target_w), 255, dtype=np.uint8)

        y_offset = (target_h - new_h) // 2
        x_offset = (target_w - new_w) // 2
        canvas[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = resized

        return canvas

    def standardize_image(self, image_path):
        """Complete standardization pipeline."""
        try:
            # Load with EXIF correction
            pil_img = self.correct_exif_orientation(image_path)
            pil_img = pil_img.convert("RGB")
            image = np.array(pil_img)

            # Ensure RGB
            if len(image.shape) == 2:
                image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
            elif image.shape[2] == 4:
                image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)
            elif image.shape[2] == 3:
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            # Resize
            image = self.resize_with_padding(image, self.target_size)

            return image, True
        except Exception as e:
            logger.error(f"Failed to standardize {image_path}: {e}")
            return None, False

    def save_image(self, image, output_path):
        """Save image as JPEG."""
        try:
            output_path = Path(output_path)
            output_path.parent.mkdir(parents=True, exist_ok=True)

            # Convert RGB to BGR for OpenCV saving (actually BGR->RGB)
            if len(image.shape) == 3:
                image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            else:
                image_rgb = image

            pil_img = Image.fromarray(image_rgb)
            pil_img.save(output_path, "JPEG", quality=self.jpeg_quality, optimize=True)
            return True
        except Exception as e:
            logger.error(f"Failed to save {output_path}: {e}")
            return False

print("✓ Image Processor loaded")

✓ Image Processor loaded


In [12]:
class MetadataGenerator:
    """Generate and manage comprehensive image metadata."""

    COLUMNS = [
        "image_id", "image_name", "disease_class", "contributor_name", "device_type",
        "original_path", "original_format", "original_size_mb",
        "processed_path", "processed_format", "processed_size_mb",
        "is_quality", "blur_score", "brightness_score", "contrast_score",
        "is_duplicate", "quality_flags", "split_assignment"
    ]

    def __init__(self):
        self.records = []
        self.counter = 0

    def create_record(self, **kwargs):
        """Create metadata record for single image."""
        self.counter += 1
        record = {
            "image_id": f"IMG_{self.counter:06d}",
            "image_name": kwargs.get("image_name"),
            "disease_class": kwargs.get("disease_class"),
            "contributor_name": kwargs.get("contributor_name"),
            "device_type": kwargs.get("device_type"),
            "original_path": str(kwargs.get("original_path")),
            "original_format": kwargs.get("original_format"),
            "original_size_mb": kwargs.get("original_size_mb", 0),
            "processed_path": str(kwargs.get("processed_path")) if kwargs.get("processed_path") else None,
            "processed_format": kwargs.get("processed_format"),
            "processed_size_mb": kwargs.get("processed_size_mb", 0),
            "is_quality": kwargs.get("is_quality", False),
            "blur_score": kwargs.get("blur_score"),
            "brightness_score": kwargs.get("brightness_score"),
            "contrast_score": kwargs.get("contrast_score"),
            "is_duplicate": kwargs.get("is_duplicate", False),
            "quality_flags": kwargs.get("quality_flags", ""),
            "split_assignment": "unassigned"
        }
        self.records.append(record)
        return record

    def to_dataframe(self):
        """Convert to pandas DataFrame."""
        df = pd.DataFrame(self.records)
        for col in self.COLUMNS:
            if col not in df.columns:
                df[col] = None
        return df[self.COLUMNS]

    def save_csv(self, path):
        """Save metadata to CSV."""
        df = self.to_dataframe()
        df.to_csv(path, index=False)
        logger.info(f"Saved {len(df)} records to {path}")
        return df

print("✓ Metadata Generator loaded")

# Initialize components
image_processor = ImageProcessor(Config.TARGET_IMAGE_SIZE, Config.JPEG_QUALITY)
quality_checker = QualityChecker()
duplicate_detector = DuplicateDetector(
    hash_size=Config.HASH_SIZE,
    algorithm=Config.HASH_ALGORITHM,
    threshold=Config.DUPLICATE_THRESHOLD
)
metadata_generator = MetadataGenerator()

✓ Metadata Generator loaded


In [13]:
def discover_images(dataset_root, config=Config):
    """Discover all images in dataset structure."""
    images_structure = defaultdict(lambda: defaultdict(list))

    for disease_folder in dataset_root.iterdir():
        if not disease_folder.is_dir():
            continue

        disease_name = disease_folder.name
        if disease_name not in config.DISEASE_CLASSES:
            continue

        for contributor_folder in disease_folder.iterdir():
            if not contributor_folder.is_dir():
                continue

            contributor_name = contributor_folder.name

            for image_file in contributor_folder.rglob("*"):
                if image_file.is_file():
                    ext = image_file.suffix.lower()
                    if ext in config.SUPPORTED_FORMATS:
                        images_structure[disease_name][contributor_name].append(image_file)

    return dict(images_structure)

def get_image_summary(images_structure):
    """Generate summary of discovered images."""
    summary = {"total_images": 0, "diseases": {}}

    for disease, contributors in images_structure.items():
        disease_total = 0
        for images in contributors.values():
            disease_total += len(images)

        summary["diseases"][disease] = {
            "total": disease_total,
            "contributors": {c: len(imgs) for c, imgs in contributors.items()}
        }
        summary["total_images"] += disease_total

    return summary

# Discover images
logger.info("Discovering images from dataset...")
images_structure = discover_images(dataset_root)
summary = get_image_summary(images_structure)

logger.info(f"✓ Found {summary['total_images']} images")
for disease, data in summary['diseases'].items():
    logger.info(f"  {disease}: {data['total']} images")

2026-04-18 15:48:17 - preprocessing - INFO - Discovering images from dataset...
INFO:preprocessing:Discovering images from dataset...
2026-04-18 15:48:18 - preprocessing - INFO - ✓ Found 1211 images
INFO:preprocessing:✓ Found 1211 images
2026-04-18 15:48:18 - preprocessing - INFO -   healthy_leaves: 480 images
INFO:preprocessing:  healthy_leaves: 480 images
2026-04-18 15:48:18 - preprocessing - INFO -   panama_wilt: 90 images
INFO:preprocessing:  panama_wilt: 90 images
2026-04-18 15:48:18 - preprocessing - INFO -   potassium_deficiency: 254 images
INFO:preprocessing:  potassium_deficiency: 254 images
2026-04-18 15:48:18 - preprocessing - INFO -   sigatoka: 387 images
INFO:preprocessing:  sigatoka: 387 images


In [14]:
def get_file_size_mb(file_path):
    """Get file size in MB."""
    return Path(file_path).stat().st_size / (1024 * 1024)

def process_single_image(image_path, disease_class, contributor, config=Config):
    """Process single image through complete pipeline."""
    try:
        image_path = Path(image_path)
        device_type = config.DEVICE_MAPPING.get(contributor, "Unknown")
        original_format = image_path.suffix.lstrip('.')
        original_size_mb = get_file_size_mb(image_path)

        # Load and standardize
        image, success = image_processor.standardize_image(image_path)

        if not success:
            metadata_generator.create_record(
                image_name=image_path.name,
                disease_class=disease_class,
                contributor_name=contributor,
                device_type=device_type,
                original_path=image_path,
                original_format=original_format,
                original_size_mb=original_size_mb,
                is_quality=False,
                quality_flags="Failed to load"
            )
            return False

        # Quality check
        quality_report = quality_checker.check_image_quality(image, config)

        # Save processed image
        output_filename = image_path.stem + ".jpg"
        output_path = processed_dir / disease_class / output_filename

        save_success = image_processor.save_image(image, output_path)

        if not save_success:
            return False

        processed_size_mb = get_file_size_mb(output_path)

        # Create metadata record
        metadata_generator.create_record(
            image_name=image_path.name,
            disease_class=disease_class,
            contributor_name=contributor,
            device_type=device_type,
            original_path=image_path,
            original_format=original_format,
            original_size_mb=original_size_mb,
            processed_path=output_path,
            processed_format="JPEG",
            processed_size_mb=processed_size_mb,
            is_quality=quality_report["is_quality"],
            blur_score=quality_report["blur_score"],
            brightness_score=quality_report["brightness"],
            contrast_score=quality_report["contrast"],
            quality_flags=quality_report["flags"]
        )

        return True

    except Exception as e:
        logger.error(f"Error processing {image_path}: {e}")
        return False

# Process all images
logger.info("\nProcessing all images...")
all_images = []
for disease, contributors in images_structure.items():
    for contributor, images in contributors.items():
        all_images.extend([(img, disease, contributor) for img in images])

processed_count = 0
failed_count = 0

for i, (image_path, disease, contributor) in enumerate(all_images):
    if i % max(1, len(all_images) // 20) == 0:
        logger.info(f"Progress: {i}/{len(all_images)} ({i*100//len(all_images)}%)")

    if process_single_image(image_path, disease, contributor):
        processed_count += 1
    else:
        failed_count += 1

logger.info(f"✓ Processed: {processed_count}, Failed: {failed_count}")

# Get metadata
metadata_df = metadata_generator.to_dataframe()
logger.info(f"Total records: {len(metadata_df)}")

2026-04-18 15:48:22 - preprocessing - INFO - 
Processing all images...
INFO:preprocessing:
Processing all images...
2026-04-18 15:48:22 - preprocessing - INFO - Progress: 0/1211 (0%)
INFO:preprocessing:Progress: 0/1211 (0%)
2026-04-18 15:48:58 - preprocessing - INFO - Progress: 60/1211 (4%)
INFO:preprocessing:Progress: 60/1211 (4%)
2026-04-18 15:49:16 - preprocessing - INFO - Progress: 120/1211 (9%)
INFO:preprocessing:Progress: 120/1211 (9%)
2026-04-18 15:49:47 - preprocessing - INFO - Progress: 180/1211 (14%)
INFO:preprocessing:Progress: 180/1211 (14%)
2026-04-18 15:50:55 - preprocessing - INFO - Progress: 240/1211 (19%)
INFO:preprocessing:Progress: 240/1211 (19%)
2026-04-18 15:51:35 - preprocessing - INFO - Progress: 300/1211 (24%)
INFO:preprocessing:Progress: 300/1211 (24%)
2026-04-18 15:52:42 - preprocessing - INFO - Progress: 360/1211 (29%)
INFO:preprocessing:Progress: 360/1211 (29%)
2026-04-18 15:54:09 - preprocessing - INFO - Progress: 420/1211 (34%)
INFO:preprocessing:Progress:

In [15]:
# Filter to quality images only
quality_metadata = metadata_df[metadata_df["is_quality"] == True].reset_index(drop=True)
logger.info(f"Quality images: {len(quality_metadata)}/{len(metadata_df)} ({len(quality_metadata)*100//len(metadata_df)}%)")

# Detect duplicates
logger.info("\nDetecting duplicate images...")
processed_paths = quality_metadata["processed_path"].dropna().tolist()
duplicate_results = duplicate_detector.find_duplicates_in_batch(processed_paths)

logger.info(f"Found {duplicate_results['duplicate_count']} duplicates in {len(duplicate_results['duplicate_groups'])} groups")
logger.info(f"Unique images: {duplicate_results['unique_count']}")

# Mark duplicates in metadata
duplicate_set = set()
for group in duplicate_results['duplicate_groups']:
    for img_path in group:
        duplicate_set.add(img_path)

quality_metadata["is_duplicate"] = quality_metadata["processed_path"].isin(duplicate_set)

# Remove duplicates, keep only first of each group
keep_indices = []
seen_hashes = set()

for idx, row in quality_metadata.iterrows():
    if row["is_duplicate"]:
        # Check if we've seen this duplicate group before
        continue
    keep_indices.append(idx)

unique_metadata = quality_metadata.iloc[keep_indices].reset_index(drop=True)
logger.info(f"After removing duplicates: {len(unique_metadata)} images remain")

2026-04-18 16:17:58 - preprocessing - INFO - Quality images: 1211/1211 (100%)
INFO:preprocessing:Quality images: 1211/1211 (100%)
2026-04-18 16:17:58 - preprocessing - INFO - 
Detecting duplicate images...
INFO:preprocessing:
Detecting duplicate images...
2026-04-18 16:18:13 - preprocessing - INFO - Found 102 duplicates in 76 groups
INFO:preprocessing:Found 102 duplicates in 76 groups
2026-04-18 16:18:13 - preprocessing - INFO - Unique images: 1109
INFO:preprocessing:Unique images: 1109
2026-04-18 16:18:13 - preprocessing - INFO - After removing duplicates: 1033 images remain
INFO:preprocessing:After removing duplicates: 1033 images remain


In [16]:
def stratified_split(data_df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    """
    Split dataset with stratification by disease class.
    WHY: Maintains disease class proportions in each split.
    """
    # First split: train+val vs test
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_seed)
    train_val_idx, test_idx = next(splitter.split(data_df, data_df["disease_class"]))

    train_val_df = data_df.iloc[train_val_idx].reset_index(drop=True)
    test_df = data_df.iloc[test_idx].reset_index(drop=True)

    # Second split: train vs val
    val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
    splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio_adjusted, random_state=random_seed)
    train_idx, val_idx = next(splitter_val.split(train_val_df, train_val_df["disease_class"]))

    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    return {"train": train_df, "val": val_df, "test": test_df}

# Perform stratified splitting
logger.info("\nPerforming stratified split...")
splits = stratified_split(
    unique_metadata,
    train_ratio=Config.TRAIN_RATIO,
    val_ratio=Config.VAL_RATIO,
    test_ratio=Config.TEST_RATIO,
    random_seed=Config.RANDOM_SEED
)

logger.info(f"Train set: {len(splits['train'])} images ({len(splits['train'])*100//len(unique_metadata):.1f}%)")
logger.info(f"Val set: {len(splits['val'])} images ({len(splits['val'])*100//len(unique_metadata):.1f}%)")
logger.info(f"Test set: {len(splits['test'])} images ({len(splits['test'])*100//len(unique_metadata):.1f}%)")

# Verify stratification
for split_name, split_df in splits.items():
    dist = split_df["disease_class"].value_counts(normalize=True)
    logger.info(f"\n{split_name.upper()} distribution:")
    for disease, pct in dist.items():
        logger.info(f"  {disease}: {pct*100:.1f}%")

2026-04-18 16:19:03 - preprocessing - INFO - 
Performing stratified split...
INFO:preprocessing:
Performing stratified split...
2026-04-18 16:19:04 - preprocessing - INFO - Train set: 723 images (69.0%)
INFO:preprocessing:Train set: 723 images (69.0%)
2026-04-18 16:19:04 - preprocessing - INFO - Val set: 155 images (15.0%)
INFO:preprocessing:Val set: 155 images (15.0%)
2026-04-18 16:19:04 - preprocessing - INFO - Test set: 155 images (15.0%)
INFO:preprocessing:Test set: 155 images (15.0%)
2026-04-18 16:19:04 - preprocessing - INFO - 
TRAIN distribution:
INFO:preprocessing:
TRAIN distribution:
2026-04-18 16:19:04 - preprocessing - INFO -   healthy_leaves: 37.9%
INFO:preprocessing:  healthy_leaves: 37.9%
2026-04-18 16:19:04 - preprocessing - INFO -   sigatoka: 33.9%
INFO:preprocessing:  sigatoka: 33.9%
2026-04-18 16:19:04 - preprocessing - INFO -   potassium_deficiency: 21.4%
INFO:preprocessing:  potassium_deficiency: 21.4%
2026-04-18 16:19:04 - preprocessing - INFO -   panama_wilt: 6.8%

In [18]:
# Create cross-device validation splits
logger.info("\nCreating cross-device validation splits...")
logger.info("WHY: Verify model learns disease patterns, not phone characteristics")

devices = unique_metadata["device_type"].unique()
logger.info(f"Devices in dataset: {devices}")

# Cross-device splits
iphone_data = unique_metadata[unique_metadata["device_type"] == "iPhone"].reset_index(drop=True)
android_data = unique_metadata[unique_metadata["device_type"] == "Android"].reset_index(drop=True)

logger.info(f"\nCross-Device Split Details:")
logger.info(f"  iPhone images: {len(iphone_data)}")
logger.info(f"  Android images: {len(android_data)}")

# If both devices present
if len(iphone_data) > 0 and len(android_data) > 0:
    logger.info(f"\n✓ Cross-device validation possible!")
    logger.info(f"  Will test: Train on iPhone (or Android) -> Test on Android (or iPhone)")
else:
    logger.warning(f"⚠ Insufficient device diversity for strong cross-device validation")

# Assign splits to metadata
for split_name, split_df in splits.items():
    split_df["split_assignment"] = split_name

splits["train"] = splits["train"].copy()
splits["val"] = splits["val"].copy()
splits["test"] = splits["test"].copy()

splits["train"]["split_assignment"] = "train"
splits["val"]["split_assignment"] = "val"
splits["test"]["split_assignment"] = "test"

complete_metadata = pd.concat([splits["train"], splits["val"], splits["test"]], ignore_index=True)

# Actually assign using index matching
split_indices = {}
for split_name, split_df in splits.items():
    split_indices[split_name] = set(range(len(unique_metadata)))  # Simplified

# Better approach: use original indices
for split_name, split_df in splits.items():
    indices = split_df.index.tolist()
    unique_metadata.loc[indices, "split_assignment"] = split_name

logger.info("\n✓ Dataset split complete")

2026-04-18 16:22:11 - preprocessing - INFO - 
Creating cross-device validation splits...
INFO:preprocessing:
Creating cross-device validation splits...
2026-04-18 16:22:11 - preprocessing - INFO - WHY: Verify model learns disease patterns, not phone characteristics
INFO:preprocessing:WHY: Verify model learns disease patterns, not phone characteristics
2026-04-18 16:22:12 - preprocessing - INFO - Devices in dataset: ['Android' 'iPhone' 'Unknown']
INFO:preprocessing:Devices in dataset: ['Android' 'iPhone' 'Unknown']
2026-04-18 16:22:12 - preprocessing - INFO - 
Cross-Device Split Details:
INFO:preprocessing:
Cross-Device Split Details:
2026-04-18 16:22:12 - preprocessing - INFO -   iPhone images: 130
INFO:preprocessing:  iPhone images: 130
2026-04-18 16:22:12 - preprocessing - INFO -   Android images: 639
INFO:preprocessing:  Android images: 639
2026-04-18 16:22:12 - preprocessing - INFO - 
✓ Cross-device validation possible!
INFO:preprocessing:
✓ Cross-device validation possible!
2026-0

In [19]:
# Augmentation configuration (not applied during preprocessing)
# Applied during model training instead
AUGMENTATION_CONFIG = {
    "rotation_range": 20,           # Degrees
    "zoom_range": [0.8, 1.2],       # 80-120% zoom
    "brightness_range": [0.8, 1.2], # 80-120% brightness
    "horizontal_flip": True,        # Allow horizontal flip
    "vertical_flip": False,         # Don't flip vertically (most leaves grow up)
    "random_crop_size": None,       # Set to (h, w) to enable
}

logger.info("\n✓ Augmentation configuration loaded")
logger.info(f"  During training, will apply:")
for key, value in AUGMENTATION_CONFIG.items():
    logger.info(f"    - {key}: {value}")

print("\nAugmentation note: These will be applied during model training, not preprocessing.")

2026-04-18 16:22:15 - preprocessing - INFO - 
✓ Augmentation configuration loaded
INFO:preprocessing:
✓ Augmentation configuration loaded
2026-04-18 16:22:15 - preprocessing - INFO -   During training, will apply:
INFO:preprocessing:  During training, will apply:
2026-04-18 16:22:15 - preprocessing - INFO -     - rotation_range: 20
INFO:preprocessing:    - rotation_range: 20
2026-04-18 16:22:15 - preprocessing - INFO -     - zoom_range: [0.8, 1.2]
INFO:preprocessing:    - zoom_range: [0.8, 1.2]
2026-04-18 16:22:15 - preprocessing - INFO -     - brightness_range: [0.8, 1.2]
INFO:preprocessing:    - brightness_range: [0.8, 1.2]
2026-04-18 16:22:15 - preprocessing - INFO -     - horizontal_flip: True
INFO:preprocessing:    - horizontal_flip: True
2026-04-18 16:22:15 - preprocessing - INFO -     - vertical_flip: False
INFO:preprocessing:    - vertical_flip: False
2026-04-18 16:22:15 - preprocessing - INFO -     - random_crop_size: None
INFO:preprocessing:    - random_crop_size: None



Augmentation note: These will be applied during model training, not preprocessing.


In [20]:
logger.info("\n" + "="*70)
logger.info("DATASET QUALITY REPORT")
logger.info("="*70)

# Overall statistics
total_original = len(metadata_df)
quality_images = metadata_df["is_quality"].sum()
quality_rate = quality_images / total_original * 100 if total_original > 0 else 0

logger.info(f"\n1. IMAGE QUALITY FILTERING:")
logger.info(f"  Original images discovered: {total_original}")
logger.info(f"  Quality images retained: {quality_images} ({quality_rate:.1f}%)")
logger.info(f"  Rejected images: {total_original - quality_images} ({100-quality_rate:.1f}%)")

# Reason breakdown
reject_reasons = []
for _, row in metadata_df[metadata_df["is_quality"] == False].iterrows():
    if row["quality_flags"]:
        reject_reasons.extend(row["quality_flags"].split(";"))

from collections import Counter
reason_counts = Counter(reject_reasons)
logger.info(f"\n  Rejection reasons:")
for reason, count in reason_counts.most_common(5):
    logger.info(f"    - {reason}: {count}")

# Duplicate detection
logger.info(f"\n2. DUPLICATE DETECTION:")
logger.info(f"  Unique images in quality set: {len(quality_metadata)}")
logger.info(f"  Duplicate groups found: {len(duplicate_results['duplicate_groups'])}")
logger.info(f"  Duplicate images removed: {duplicate_results['duplicate_count']}")
logger.info(f"  Final unique images: {len(unique_metadata)}")

# Split distribution
logger.info(f"\n3. DATASET SPLITS:")
logger.info(f"  Train set: {len(splits['train'])} ({len(splits['train'])*100//len(unique_metadata):.1f}%)")
logger.info(f"  Val set: {len(splits['val'])} ({len(splits['val'])*100//len(unique_metadata):.1f}%)")
logger.info(f"  Test set: {len(splits['test'])} ({len(splits['test'])*100//len(unique_metadata):.1f}%)")

# Device distribution
logger.info(f"\n4. DEVICE DISTRIBUTION:")
device_dist = unique_metadata["device_type"].value_counts()
for device, count in device_dist.items():
    pct = count * 100 / len(unique_metadata)
    logger.info(f"  {device}: {count} ({pct:.1f}%)")

# Disease distribution
logger.info(f"\n5. DISEASE CLASS DISTRIBUTION:")
disease_dist = unique_metadata["disease_class"].value_counts()
for disease, count in disease_dist.items():
    pct = count * 100 / len(unique_metadata)
    logger.info(f"  {disease}: {count} ({pct:.1f}%)")

# Quality metrics
logger.info(f"\n6. QUALITY METRICS:")
if "blur_score" in unique_metadata.columns:
    logger.info(f"  Blur Score - Mean: {unique_metadata['blur_score'].mean():.1f}, Std: {unique_metadata['blur_score'].std():.1f}")
if "brightness_score" in unique_metadata.columns:
    logger.info(f"  Brightness - Mean: {unique_metadata['brightness_score'].mean():.1f}, Std: {unique_metadata['brightness_score'].std():.1f}")
if "contrast_score" in unique_metadata.columns:
    logger.info(f"  Contrast - Mean: {unique_metadata['contrast_score'].mean():.1f}, Std: {unique_metadata['contrast_score'].std():.1f}")

logger.info("\n" + "="*70)

2026-04-18 16:22:18 - preprocessing - INFO - 
INFO:preprocessing:
2026-04-18 16:22:18 - preprocessing - INFO - DATASET QUALITY REPORT
INFO:preprocessing:DATASET QUALITY REPORT
2026-04-18 16:22:18 - preprocessing - INFO - ======================================================================
INFO:preprocessing:======================================================================
2026-04-18 16:22:18 - preprocessing - INFO - 
1. IMAGE QUALITY FILTERING:
INFO:preprocessing:
1. IMAGE QUALITY FILTERING:
2026-04-18 16:22:18 - preprocessing - INFO -   Original images discovered: 1211
INFO:preprocessing:  Original images discovered: 1211
2026-04-18 16:22:18 - preprocessing - INFO -   Quality images retained: 1211 (100.0%)
INFO:preprocessing:  Quality images retained: 1211 (100.0%)
2026-04-18 16:22:18 - preprocessing - INFO -   Rejected images: 0 (0.0%)
INFO:preprocessing:  Rejected images: 0 (0.0%)
2026-04-18 16:22:18 - preprocessing - INFO - 
  Rejection reasons:
INFO:preprocessing:
  Rejecti

In [21]:
# Save metadata and splits
logger.info("\nSaving results...")

# Save main metadata
metadata_csv_path = metadata_dir / "image_metadata.csv"
unique_metadata.to_csv(metadata_csv_path, index=False)
logger.info(f"✓ Main metadata: {metadata_csv_path}")

# Save split assignments
split_csv_path = metadata_dir / "splits.csv"
split_data = []
for split_name, split_df in splits.items():
    for _, row in split_df.iterrows():
        split_data.append({
            "image_id": row["image_id"],
            "image_name": row["image_name"],
            "disease_class": row["disease_class"],
            "split": split_name
        })
split_df_csv = pd.DataFrame(split_data)
split_df_csv.to_csv(split_csv_path, index=False)
logger.info(f"✓ Split assignments: {split_csv_path}")

# Save configuration
config_json_path = metadata_dir / "preprocessing_config.json"
config_dict = {
    "target_image_size": Config.TARGET_IMAGE_SIZE,
    "jpeg_quality": Config.JPEG_QUALITY,
    "blur_threshold": Config.BLUR_THRESHOLD,
    "brightness_range": [Config.MIN_BRIGHTNESS, Config.MAX_BRIGHTNESS],
    "duplicate_threshold": Config.DUPLICATE_THRESHOLD,
    "train_ratio": Config.TRAIN_RATIO,
    "val_ratio": Config.VAL_RATIO,
    "test_ratio": Config.TEST_RATIO,
    "random_seed": Config.RANDOM_SEED,
    "hash_algorithm": Config.HASH_ALGORITHM,
    "augmentation_config": AUGMENTATION_CONFIG
}
with open(config_json_path, 'w') as f:
    json.dump(config_dict, f, indent=2)
logger.info(f"✓ Configuration: {config_json_path}")

# Save statistics report
stats_report = {
    "original_images": total_original,
    "quality_images": quality_images,
    "quality_rate": quality_rate,
    "unique_images_after_dedup": len(unique_metadata),
    "duplicate_count": duplicate_results['duplicate_count'],
    "splits": {
        "train": len(splits['train']),
        "val": len(splits['val']),
        "test": len(splits['test'])
    },
    "disease_distribution": disease_dist.to_dict(),
    "device_distribution": device_dist.to_dict(),
    "timestamp": datetime.now().isoformat()
}
stats_json_path = metadata_dir / "dataset_statistics.json"
with open(stats_json_path, 'w') as f:
    json.dump(stats_report, f, indent=2, default=str)
logger.info(f"✓ Statistics: {stats_json_path}")

logger.info(f"\n✓ All results saved to: {metadata_dir}")

# Display file structure
logger.info("\nOutput directory structure:")
logger.info(f"{str(output_root)}/")
logger.info(f"  processed/")
for disease in Config.DISEASE_CLASSES:
    count = len(list((processed_dir / disease).glob("*.jpg")))
    logger.info(f"    {disease}/ -> {count} images")

logger.info(f"  split/")
logger.info(f"    train/ -> {len(splits['train'])} images")
logger.info(f"    val/ -> {len(splits['val'])} images")
logger.info(f"    test/ -> {len(splits['test'])} images")

logger.info(f"  metadata/")
logger.info(f"    image_metadata.csv")
logger.info(f"    splits.csv")
logger.info(f"    preprocessing_config.json")
logger.info(f"    dataset_statistics.json")

2026-04-18 16:22:24 - preprocessing - INFO - 
Saving results...
INFO:preprocessing:
Saving results...
2026-04-18 16:22:24 - preprocessing - INFO - ✓ Main metadata: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/image_metadata.csv
INFO:preprocessing:✓ Main metadata: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/image_metadata.csv
2026-04-18 16:22:24 - preprocessing - INFO - ✓ Split assignments: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/splits.csv
INFO:preprocessing:✓ Split assignments: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/splits.csv
2026-04-18 16:22:24 - preprocessing - INFO - ✓ Configuration: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/preprocessing_config.json
INFO:preprocessing:✓ Configuration: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/preprocessing_config.json
2026-04-18 16:22:24 - preprocessing - INFO - ✓ Statistics: /content/drive/MyDrive/Banana_Leaf_Processed/metadata/dataset_statistics.json
INFO:preprocessing:✓ St

In [22]:
logger.info("\n" + "="*70)
logger.info("PREPROCESSING COMPLETE!")
logger.info("="*70)

logger.info(f"""
NEXT STEPS:

1. VERIFY DATASET QUALITY:
   - Review {metadata_csv_path}
   - Check for any unexpected quality flags
   - Verify class distributions are balanced

2. CROSS-DEVICE VALIDATION SETUP:
   - Train on iPhone images: {len(iphone_data) if len(iphone_data) > 0 else 'N/A'} images
   - Train on Android images: {len(android_data) if len(android_data) > 0 else 'N/A'} images
   - This tests if model learns disease patterns or device characteristics

3. BUILD CNN MODEL:
   - Use images from dataset/processed/ or dataset/split/
   - Input size: {Config.TARGET_IMAGE_SIZE}
   - Input format: RGB JPEG
   - Classes: {len(Config.DISEASE_CLASSES)} ({', '.join(Config.DISEASE_CLASSES)})

4. TRAINING TIPS:
   - Apply augmentation during training (not preprocessing)
   - Use train/ set only for training
   - Tune hyperparameters on val/ set
   - Evaluate on test/ set (never during training)
   - Compare cross-device accuracy to detect bias

5. IMPORTANT: AVOID DATA LEAKAGE:
   - Never use test set during training
   - Never use test set for hyperparameter tuning
   - Report final accuracy on test set only
   - Cross-device accuracy is your TRUE accuracy metric

Dataset ready for model training! 🎉
""")

2026-04-18 16:22:28 - preprocessing - INFO - 
INFO:preprocessing:
2026-04-18 16:22:28 - preprocessing - INFO - PREPROCESSING COMPLETE!
INFO:preprocessing:PREPROCESSING COMPLETE!
2026-04-18 16:22:28 - preprocessing - INFO - ======================================================================
INFO:preprocessing:======================================================================
2026-04-18 16:22:28 - preprocessing - INFO - 
NEXT STEPS:

1. VERIFY DATASET QUALITY:
   - Review /content/drive/MyDrive/Banana_Leaf_Processed/metadata/image_metadata.csv
   - Check for any unexpected quality flags
   - Verify class distributions are balanced

2. CROSS-DEVICE VALIDATION SETUP:
   - Train on iPhone images: 130 images
   - Train on Android images: 639 images
   - This tests if model learns disease patterns or device characteristics

3. BUILD CNN MODEL:
   - Use images from dataset/processed/ or dataset/split/
   - Input size: (256, 256)
   - Input format: RGB JPEG
   - Classes: 4 (healthy_leave

In [23]:
import os
base = "/content/drive/MyDrive/Banana_Leaf_Processed"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 2:  # only show files 2 levels deep
        for f in files[:3]:  # show first 3 files per folder
            print(f"{indent}  {f}")

Banana_Leaf_Processed/
  processed/
    healthy_leaves/
    panama_wilt/
    potassium_deficiency/
    sigatoka/
  split/
    train/
      healthy_leaves/
      panama_wilt/
      potassium_deficiency/
      sigatoka/
    val/
      healthy_leaves/
      panama_wilt/
      potassium_deficiency/
      sigatoka/
    test/
      healthy_leaves/
      panama_wilt/
      potassium_deficiency/
      sigatoka/
  metadata/
    image_metadata.csv
    splits.csv
    preprocessing_config.json


In [24]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
import os

base = "/content/drive/MyDrive/Banana_Leaf_Processed"
for folder in ["processed/healthy_leaves", "split/train/healthy_leaves"]:
    path = os.path.join(base, folder)
    files = os.listdir(path)
    print(f"{folder}: {len(files)} files")

processed/healthy_leaves: 480 files
split/train/healthy_leaves: 0 files


In [26]:
import os

base = "/content/drive/MyDrive/Banana_Leaf_Processed"
classes = ["healthy_leaves", "panama_wilt", "potassium_deficiency", "sigatoka"]
splits = ["train", "val", "test"]

for split in splits:
    total = 0
    for cls in classes:
        path = os.path.join(base, "split", split, cls)
        count = len(os.listdir(path)) if os.path.exists(path) else 0
        total += count
        print(f"  {split}/{cls}: {count} files")
    print(f"  → {split} total: {total}\n")

  train/healthy_leaves: 0 files
  train/panama_wilt: 0 files
  train/potassium_deficiency: 0 files
  train/sigatoka: 0 files
  → train total: 0

  val/healthy_leaves: 0 files
  val/panama_wilt: 0 files
  val/potassium_deficiency: 0 files
  val/sigatoka: 0 files
  → val total: 0

  test/healthy_leaves: 0 files
  test/panama_wilt: 0 files
  test/potassium_deficiency: 0 files
  test/sigatoka: 0 files
  → test total: 0



In [27]:
import shutil
import os
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Banana_Leaf_Processed")
split_dir = base / "split"

# Load the metadata which has processed_path and split_assignment
df = pd.read_csv(base / "metadata/image_metadata.csv")

print(f"Total rows in metadata: {len(df)}")
print(df["split_assignment"].value_counts())
print(df[["processed_path", "split_assignment", "disease_class"]].head(3))

Total rows in metadata: 1033
split_assignment
train         568
unassigned    310
test          155
Name: count, dtype: int64
                                      processed_path split_assignment  \
0  /content/drive/MyDrive/Banana_Leaf_Processed/p...             test   
1  /content/drive/MyDrive/Banana_Leaf_Processed/p...             test   
2  /content/drive/MyDrive/Banana_Leaf_Processed/p...             test   

    disease_class  
0  healthy_leaves  
1  healthy_leaves  
2  healthy_leaves  


In [30]:
copied = 0
skipped = 0
valid_splits = ["train", "val", "test"]

for _, row in df.iterrows():
    src = row["processed_path"]
    split = row["split_assignment"]
    disease = row["disease_class"]

    # Skip if missing or not a valid split
    if pd.isna(src) or pd.isna(split) or pd.isna(disease):
        skipped += 1
        continue

    if split not in valid_splits:
        skipped += 1
        continue

    src_path = Path(src)
    dst_path = split_dir / split / disease / src_path.name

    if src_path.exists():
        shutil.copy2(src_path, dst_path)
        copied += 1
    else:
        skipped += 1

print(f"✓ Copied: {copied} files")
print(f"✗ Skipped: {skipped} files")

✓ Copied: 723 files
✗ Skipped: 310 files


In [29]:
import shutil
import os
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Banana_Leaf_Processed")
split_dir = base / "split"

df = pd.read_csv(base / "metadata/image_metadata.csv")

# Check what split values exist
print(df["split_assignment"].value_counts())

split_assignment
train         568
unassigned    310
test          155
Name: count, dtype: int64


In [31]:
print(df["split_assignment"].value_counts())

split_assignment
train         568
unassigned    310
test          155
Name: count, dtype: int64


In [35]:
def stratified_split(data_df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=random_seed)
    train_val_idx, test_idx = next(splitter.split(data_df, data_df["disease_class"]))
    train_val_df = data_df.iloc[train_val_idx].reset_index(drop=True)
    test_df = data_df.iloc[test_idx].reset_index(drop=True)

    val_ratio_adjusted = val_ratio / (train_ratio + val_ratio)
    splitter_val = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio_adjusted, random_state=random_seed)
    train_idx, val_idx = next(splitter_val.split(train_val_df, train_val_df["disease_class"]))

    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    return {"train": train_df, "val": val_df, "test": test_df}

splits = stratified_split(unique_metadata)
print(f"Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])}")

Train: 723, Val: 155, Test: 155


In [36]:
import shutil
import pandas as pd
from pathlib import Path

base = Path("/content/drive/MyDrive/Banana_Leaf_Processed")
split_dir = base / "split"

copied = 0
skipped = 0

for split_name, split_df in splits.items():
    for _, row in split_df.iterrows():
        src = row.get("processed_path")
        disease = row.get("disease_class")

        if pd.isna(src) or pd.isna(disease):
            skipped += 1
            continue

        src_path = Path(src)
        dst_path = split_dir / split_name / disease / src_path.name

        if src_path.exists():
            shutil.copy2(src_path, dst_path)
            copied += 1
        else:
            skipped += 1

print(f"✓ Copied: {copied} files")
print(f"✗ Skipped: {skipped} files")

✓ Copied: 1033 files
✗ Skipped: 0 files


In [37]:
import os
for split in ["train", "val", "test"]:
    for cls in ["healthy_leaves", "panama_wilt", "potassium_deficiency", "sigatoka"]:
        path = f"/content/drive/MyDrive/Banana_Leaf_Processed/split/{split}/{cls}"
        print(f"  {split}/{cls}: {len(os.listdir(path))} files")

  train/healthy_leaves: 353 files
  train/panama_wilt: 70 files
  train/potassium_deficiency: 221 files
  train/sigatoka: 258 files
  val/healthy_leaves: 59 files
  val/panama_wilt: 11 files
  val/potassium_deficiency: 33 files
  val/sigatoka: 52 files
  test/healthy_leaves: 192 files
  test/panama_wilt: 10 files
  test/potassium_deficiency: 33 files
  test/sigatoka: 53 files


## 🎯 NEXT: Run Hybrid CNN Training with Data Augmentation

Your dataset is now preprocessed and ready! Follow these steps to train the hybrid CNN model with data augmentation in Google Colab:

### Step 1: Create New Notebook in Colab
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Click **File** → **New notebook**
3. Rename it to `hybrid_cnn_training.ipynb`

### Step 2: Mount Google Drive (Cell 1)
```python
from google.colab import drive
drive.mount('/content/drive')
```

### Step 3: Install Required Libraries (Cell 2)
```python
!pip install -q tensorflow keras numpy pandas scikit-learn pillow opencv-python matplotlib seaborn
```

### Step 4: Define Data Paths (Cell 3)
```python
from pathlib import Path

gdrive_root = Path('/content/drive/MyDrive')
dataset_dir = gdrive_root / 'Banana_Leaf_Processed' / 'split'
output_dir = gdrive_root / 'Model_Outputs'
output_dir.mkdir(exist_ok=True)

print(f"Dataset: {dataset_dir}")
print(f"Output: {output_dir}")
```

### Step 5: Copy & Paste the Augmented CNN Training Code Below
(See the training code in the next cells)

In [ ]:
# 1. IMPORTS & CONFIG
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
from datetime import datetime

# TensorFlow imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.optimizers import Adam

# ML utilities
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

print("✓ TensorFlow version:", tf.__version__)
print("✓ GPU available:", len(tf.config.list_physical_devices('GPU')) > 0)

# Configuration
IMG_SIZE = 224  # Standard for CNN
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 4
CLASS_NAMES = ["healthy_leaves", "panama_wilt", "potassium_deficiency", "sigatoka"]

print(f"\n✓ Configuration loaded:")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Epochs: {EPOCHS}")
print(f"  - Classes: {NUM_CLASSES}")


In [ ]:
# 2. SETUP DATA AUGMENTATION
print("Setting up data augmentation...")

# Training data augmentation - 9 transformations
train_augmentation = ImageDataGenerator(
    rotation_range=25,              # Random rotation ±25°
    zoom_range=0.2,                 # Random zoom 80-120%
    width_shift_range=0.2,          # Random horizontal shift
    height_shift_range=0.2,         # Random vertical shift
    shear_range=0.2,                # Shear transformation
    brightness_range=[0.8, 1.2],    # Brightness variation
    horizontal_flip=True,           # Flip horizontally
    vertical_flip=False,            # DON'T flip vertically (leaves grow up)
    fill_mode='nearest',            # Fill empty pixels
    rescale=1./255                  # Normalize to [0, 1]
)

# Validation data - NO augmentation (only normalize)
val_augmentation = ImageDataGenerator(rescale=1./255)

print("✓ Augmentation configured:")
print("  - Rotation: ±25°")
print("  - Zoom: 80-120%")
print("  - Brightness: 80-120%")
print("  - Horizontal flip: Yes")
print("  - Vertical flip: No")
print("  - Width/Height shift: ±20%")

# Load data from train/val folders
print("\nLoading training data...")
train_generator = train_augmentation.flow_from_directory(
    str(dataset_dir / 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print("Loading validation data...")
val_generator = val_augmentation.flow_from_directory(
    str(dataset_dir / 'val'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

print(f"\n✓ Data loaded:")
print(f"  - Training samples: {train_generator.samples}")
print(f"  - Validation samples: {val_generator.samples}")
print(f"  - Classes: {list(train_generator.class_indices.keys())}")

# Compute class weights for imbalanced data
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_generator.classes),
    y=train_generator.classes
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"\n✓ Class weights: {class_weight_dict}")


In [ ]:
# 3. BUILD HYBRID CNN MODELS

def build_custom_cnn(input_shape=(224, 224, 3), num_classes=4):
    """Build custom CNN from scratch - learns disease patterns directly."""
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Block 4
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        # Global pooling & dense layers
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def build_transfer_learning_model(input_shape=(224, 224, 3), num_classes=4):
    """Build transfer learning model - uses ImageNet pre-trained weights."""
    # Load pre-trained MobileNetV2 (faster, lighter, mobile-deployable)
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model weights (keep ImageNet features)
    base_model.trainable = False
    
    # Add custom top layers
    model = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model, base_model

# Build both models
print("Building Custom CNN...")
model_custom = build_custom_cnn()
print(f"✓ Custom CNN: {model_custom.count_params():,} parameters")

print("\nBuilding Transfer Learning Model (MobileNetV2)...")
model_transfer, base_model = build_transfer_learning_model()
print(f"✓ Transfer Learning: {model_transfer.count_params():,} parameters")

# Display architectures
print("\n" + "="*70)
print("CUSTOM CNN ARCHITECTURE:")
print("="*70)
model_custom.summary()

print("\n" + "="*70)
print("TRANSFER LEARNING ARCHITECTURE:")
print("="*70)
model_transfer.summary()


In [ ]:
# 4. COMPILE BOTH MODELS

print("Compiling models...")

# Optimizer with learning rate
optimizer = Adam(learning_rate=LEARNING_RATE)

# Compile Custom CNN
model_custom.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

# Compile Transfer Learning
model_transfer.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()]
)

print("✓ Both models compiled")

# Setup callbacks for training
print("\nSetting up callbacks...")

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

checkpoint_custom = callbacks.ModelCheckpoint(
    str(output_dir / 'best_custom_cnn.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

checkpoint_transfer = callbacks.ModelCheckpoint(
    str(output_dir / 'best_transfer_learning.h5'),
    monitor='val_accuracy',
    save_best_only=True,
    verbose=0
)

print("✓ Callbacks configured:")
print("  - Early Stopping (patience=10)")
print("  - Learning Rate Reduction")
print("  - Model Checkpointing")


In [ ]:
# 5. TRAIN BOTH MODELS

print("\n" + "="*70)
print("TRAINING CUSTOM CNN")
print("="*70)

history_custom = model_custom.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr, checkpoint_custom],
    verbose=1
)

print("\n" + "="*70)
print("TRAINING TRANSFER LEARNING MODEL")
print("="*70)

history_transfer = model_transfer.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=[early_stop, reduce_lr, checkpoint_transfer],
    verbose=1
)

print("\n✓ Training complete!")


In [ ]:
# 6. VISUALIZE TRAINING HISTORY

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Custom CNN - Accuracy
axes[0, 0].plot(history_custom.history['accuracy'], label='Train', linewidth=2)
axes[0, 0].plot(history_custom.history['val_accuracy'], label='Validation', linewidth=2)
axes[0, 0].set_title('Custom CNN - Accuracy', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Custom CNN - Loss
axes[0, 1].plot(history_custom.history['loss'], label='Train', linewidth=2)
axes[0, 1].plot(history_custom.history['val_loss'], label='Validation', linewidth=2)
axes[0, 1].set_title('Custom CNN - Loss', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Transfer Learning - Accuracy
axes[1, 0].plot(history_transfer.history['accuracy'], label='Train', linewidth=2)
axes[1, 0].plot(history_transfer.history['val_accuracy'], label='Validation', linewidth=2)
axes[1, 0].set_title('Transfer Learning - Accuracy', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Transfer Learning - Loss
axes[1, 1].plot(history_transfer.history['loss'], label='Train', linewidth=2)
axes[1, 1].plot(history_transfer.history['val_loss'], label='Validation', linewidth=2)
axes[1, 1].set_title('Transfer Learning - Loss', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(output_dir / 'training_history.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✓ Training history saved")


In [ ]:
# 7. EVALUATE BOTH MODELS

print("\n" + "="*70)
print("MODEL EVALUATION")
print("="*70)

def evaluate_model(model, generator, model_name):
    """Evaluate model on validation set."""
    y_true = generator.classes
    y_pred_probs = model.predict(generator, verbose=0)
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted')
    cm = confusion_matrix(y_true, y_pred)
    
    print(f"\n{model_name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'confusion_matrix': cm,
        'y_true': y_true,
        'y_pred': y_pred
    }

# Evaluate both models
results_custom = evaluate_model(model_custom, val_generator, "CUSTOM CNN")
results_transfer = evaluate_model(model_transfer, val_generator, "TRANSFER LEARNING")

# Visualize confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Custom CNN
sns.heatmap(results_custom['confusion_matrix'], annot=True, fmt='d', 
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, 
            cmap='Blues', ax=axes[0], cbar=True)
axes[0].set_title(f"Custom CNN\nAccuracy: {results_custom['accuracy']:.4f}", 
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Transfer Learning
sns.heatmap(results_transfer['confusion_matrix'], annot=True, fmt='d',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            cmap='Greens', ax=axes[1], cbar=True)
axes[1].set_title(f"Transfer Learning\nAccuracy: {results_transfer['accuracy']:.4f}",
                  fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig(str(output_dir / 'confusion_matrices.png'), dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Confusion matrices saved")


In [ ]:
# 8. ENSEMBLE PREDICTIONS (Hybrid Model Combined)

print("\n" + "="*70)
print("ENSEMBLE PREDICTION (Soft Voting)")
print("="*70)

# Get predictions from both models
y_pred_custom = model_custom.predict(val_generator, verbose=0)
y_pred_transfer = model_transfer.predict(val_generator, verbose=0)

# Soft voting: average probabilities
y_pred_ensemble = (y_pred_custom + y_pred_transfer) / 2
y_pred_ensemble_labels = np.argmax(y_pred_ensemble, axis=1)

# Evaluate ensemble
y_true = val_generator.classes
ensemble_accuracy = accuracy_score(y_true, y_pred_ensemble_labels)
ensemble_f1 = f1_score(y_true, y_pred_ensemble_labels, average='weighted')

print(f"\nENSEMBLE MODEL (Average of both):")
print(f"  Accuracy: {ensemble_accuracy:.4f}")
print(f"  F1-Score: {ensemble_f1:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_true, y_pred_ensemble_labels, target_names=CLASS_NAMES))

# Compare all three models
comparison_data = {
    'Model': ['Custom CNN', 'Transfer Learning', 'Ensemble (Hybrid)'],
    'Accuracy': [
        results_custom['accuracy'],
        results_transfer['accuracy'],
        ensemble_accuracy
    ],
    'F1-Score': [
        results_custom['f1'],
        results_transfer['f1'],
        ensemble_f1
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))

# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy comparison
axes[0].bar(comparison_df['Model'], comparison_df['Accuracy'], 
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'], edgecolor='black', linewidth=2)
axes[0].set_ylabel('Accuracy', fontsize=12, fontweight='bold')
axes[0].set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
axes[0].set_ylim([0, 1])
for i, v in enumerate(comparison_df['Accuracy']):
    axes[0].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
axes[0].grid(True, alpha=0.3, axis='y')

# F1-Score comparison
axes[1].bar(comparison_df['Model'], comparison_df['F1-Score'],
            color=['#FF6B6B', '#4ECDC4', '#45B7D1'], edgecolor='black', linewidth=2)
axes[1].set_ylabel('F1-Score', fontsize=12, fontweight='bold')
axes[1].set_title('Model F1-Score Comparison', fontsize=12, fontweight='bold')
axes[1].set_ylim([0, 1])
for i, v in enumerate(comparison_df['F1-Score']):
    axes[1].text(i, v + 0.02, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(str(output_dir / 'model_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()

print("\n✓ Model comparison saved")


In [ ]:
# 9. SAVE MODELS & CONFIGURATION

print("\nSaving models...")

# Save final models
model_custom.save(str(output_dir / 'model_custom_cnn_final.h5'))
model_transfer.save(str(output_dir / 'model_transfer_learning_final.h5'))

print(f"✓ Custom CNN saved to: {output_dir / 'model_custom_cnn_final.h5'}")
print(f"✓ Transfer Learning saved to: {output_dir / 'model_transfer_learning_final.h5'}")

# Save class mapping
class_mapping = {v: k for k, v in train_generator.class_indices.items()}
with open(output_dir / 'class_mapping.json', 'w') as f:
    json.dump(class_mapping, f, indent=2)
print(f"✓ Class mapping saved")

# Save training configuration
config = {
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'augmentation': {
        'rotation_range': 25,
        'zoom_range': 0.2,
        'brightness_range': [0.8, 1.2],
        'horizontal_flip': True,
        'vertical_flip': False
    },
    'model_custom_accuracy': float(results_custom['accuracy']),
    'model_transfer_accuracy': float(results_transfer['accuracy']),
    'ensemble_accuracy': float(ensemble_accuracy),
    'timestamp': datetime.now().isoformat()
}

with open(output_dir / 'training_config.json', 'w') as f:
    json.dump(config, f, indent=2)
print(f"✓ Training configuration saved")

# Save results summary
results_summary = {
    'custom_cnn': {
        'accuracy': float(results_custom['accuracy']),
        'f1_score': float(results_custom['f1']),
        'parameters': int(model_custom.count_params())
    },
    'transfer_learning': {
        'accuracy': float(results_transfer['accuracy']),
        'f1_score': float(results_transfer['f1']),
        'parameters': int(model_transfer.count_params())
    },
    'ensemble': {
        'accuracy': float(ensemble_accuracy),
        'f1_score': float(ensemble_f1)
    },
    'comparison_df': comparison_df.to_dict(orient='records')
}

with open(output_dir / 'results_summary.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"✓ Results summary saved")


In [ ]:
# 10. PREDICTION FUNCTIONS

from PIL import Image
import io

def preprocess_image(image_path, target_size=224):
    """Load and preprocess single image."""
    img = Image.open(image_path).convert('RGB')
    img = img.resize((target_size, target_size))
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def predict_single_model(image_path, model, class_mapping, model_name):
    """Predict using single model."""
    img_array = preprocess_image(image_path)
    prediction = model.predict(img_array, verbose=0)
    class_idx = np.argmax(prediction)
    confidence = prediction[0][class_idx]
    
    return {
        'model': model_name,
        'predicted_class': class_mapping[str(class_idx)],
        'confidence': float(confidence),
        'all_probabilities': {
            class_mapping[str(i)]: float(prediction[0][i]) 
            for i in range(len(prediction[0]))
        }
    }

def predict_ensemble(image_path, model_custom, model_transfer, class_mapping):
    """Predict using ensemble (hybrid) model."""
    img_array = preprocess_image(image_path)
    
    # Get predictions from both models
    pred_custom = model_custom.predict(img_array, verbose=0)
    pred_transfer = model_transfer.predict(img_array, verbose=0)
    
    # Soft voting
    pred_ensemble = (pred_custom + pred_transfer) / 2
    class_idx = np.argmax(pred_ensemble)
    confidence = pred_ensemble[0][class_idx]
    
    return {
        'model': 'Ensemble (Hybrid)',
        'predicted_class': class_mapping[str(class_idx)],
        'confidence': float(confidence),
        'all_probabilities': {
            class_mapping[str(i)]: float(pred_ensemble[0][i])
            for i in range(len(pred_ensemble[0]))
        },
        'custom_cnn_confidence': float(pred_custom[0][class_idx]),
        'transfer_learning_confidence': float(pred_transfer[0][class_idx])
    }

# Load class mapping
with open(output_dir / 'class_mapping.json', 'r') as f:
    class_mapping = json.load(f)

print("✓ Prediction functions ready")
print("\nTo make predictions on a single image:")
print("  result = predict_ensemble('path/to/image.jpg', model_custom, model_transfer, class_mapping)")
print("  print(result)")


In [ ]:
# 11. EXAMPLE PREDICTIONS WITH VISUALIZATION

print("Testing ensemble predictions on validation images...\n")

# Get a batch
sample_images, sample_labels = next(iter(val_generator))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i in range(min(6, len(sample_images))):
    # True label
    true_idx = np.argmax(sample_labels[i])
    true_label = class_mapping[str(true_idx)]
    
    # Display image
    axes[i].imshow(sample_images[i])
    
    # Ensemble prediction
    img_array = np.expand_dims(sample_images[i], axis=0)
    pred_custom = model_custom.predict(img_array, verbose=0)
    pred_transfer = model_transfer.predict(img_array, verbose=0)
    pred_ensemble = (pred_custom + pred_transfer) / 2
    
    pred_idx = np.argmax(pred_ensemble)
    pred_label = class_mapping[str(pred_idx)]
    pred_conf = pred_ensemble[0][pred_idx]
    
    # Title
    match = "✓" if pred_label == true_label else "✗"
    title = f"True: {true_label}\nPred: {pred_label} {match}\nConf: {pred_conf:.2f}"
    
    axes[i].set_title(title, fontsize=10, fontweight='bold',
                     color='green' if pred_label == true_label else 'red')
    axes[i].axis('off')

plt.suptitle('Ensemble Model Predictions on Validation Images', 
            fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(str(output_dir / 'example_predictions.png'), dpi=100, bbox_inches='tight')
plt.show()

print("✓ Example predictions visualized")


In [ ]:
# 12. FINAL SUMMARY REPORT

print("\n" + "="*80)
print(" " * 20 + "🎉 TRAINING COMPLETE! 🎉")
print("="*80)

print(f"""
✅ HYBRID CNN MODEL WITH DATA AUGMENTATION - COMPLETE

📊 DATASET INFORMATION:
  • Training samples: {train_generator.samples}
  • Validation samples: {val_generator.samples}
  • Classes: {NUM_CLASSES} ({', '.join(CLASS_NAMES)})
  • Image size: {IMG_SIZE}×{IMG_SIZE} pixels
  • Batch size: {BATCH_SIZE}

🔧 DATA AUGMENTATION APPLIED:
  • Rotation: ±25°
  • Zoom: 80-120%
  • Width/Height shift: ±20%
  • Brightness: 80-120%
  • Horizontal flip: Yes
  • Vertical flip: No (leaves grow upward)

🧠 MODELS TRAINED:

  1️⃣  CUSTOM CNN
     • Parameters: {model_custom.count_params():,}
     • Accuracy: {results_custom['accuracy']:.4f}
     • F1-Score: {results_custom['f1']:.4f}
     • Strength: Learns disease-specific patterns

  2️⃣  TRANSFER LEARNING (MobileNetV2)
     • Parameters: {model_transfer.count_params():,}
     • Accuracy: {results_transfer['accuracy']:.4f}
     • F1-Score: {results_transfer['f1']:.4f}
     • Strength: Better generalization, faster training

  3️⃣  ENSEMBLE (HYBRID) 🏆
     • Accuracy: {ensemble_accuracy:.4f}
     • F1-Score: {ensemble_f1:.4f}
     • Strength: Combines both models via soft voting

💾 FILES SAVED TO: {output_dir}
  ✓ model_custom_cnn_final.h5
  ✓ model_transfer_learning_final.h5
  ✓ class_mapping.json
  ✓ training_config.json
  ✓ results_summary.json
  ✓ training_history.png
  ✓ confusion_matrices.png
  ✓ model_comparison.png
  ✓ example_predictions.png

🚀 NEXT STEPS:

  1. Download saved files from your Google Drive
     - Model_Outputs/ folder contains all trained models

  2. Deploy the ensemble model:
     - Load: tf.keras.models.load_model('model_custom_cnn_final.h5')
     - Predict: Use predict_ensemble() function

  3. Fine-tune if needed:
     - Unfreeze MobileNetV2 base layers
     - Train for additional epochs with lower learning rate
     - Test on cross-device validation set

  4. Integrate into production:
     - Use ensemble predictions for best accuracy
     - Deploy on mobile/cloud (MobileNetV2 is optimized)
     - Monitor real-world performance

📈 MODEL COMPARISON:
{comparison_df.to_string(index=False)}

✨ Hybrid model ready for production! ✨
""")

print("="*80)


## 📚 COMPLETE WORKFLOW: Preprocessing → Augmentation → Hybrid CNN Training

### Overview
This notebook demonstrates the complete pipeline:
1. **Preprocessing**: Clean, validate, and split your dataset
2. **Augmentation**: Apply 9 transformations during training
3. **Hybrid CNN**: Train 2 complementary models and combine via soft voting

### Running This In Google Colab

#### **STEP 1: Create/Upload Notebooks to Google Drive**
- Save this notebook to: `My Drive/Mini_Project_Dataset/colab_pipeline.ipynb`
- The training code above is ready to run (cells 1-12)

#### **STEP 2: Open in Google Colab**
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Click **File** → **Open notebook**
3. Search for your notebook or use this link format:
   ```
   https://colab.research.google.com/drive/[FILE_ID]
   ```

#### **STEP 3: Run Preprocessing (Original Cells)**
Execute cells in order:
- ✅ Import libraries
- ✅ Mount Google Drive
- ✅ Create directory structure
- ✅ Process images (quality check, deduplication)
- ✅ Stratified split (train/val/test)
- ✅ Save results to `Banana_Leaf_Processed/`

**Runtime: ~30-60 minutes** (depends on dataset size)

#### **STEP 4: Run Hybrid CNN Training (New Cells Above)**
After preprocessing completes, run cells 1-12:
- ✅ **Cell 1**: Import & Config
- ✅ **Cell 2**: Data Augmentation Setup
- ✅ **Cell 3**: Build Models
- ✅ **Cell 4**: Compile Models
- ✅ **Cell 5**: Train Both Models ⏱️ **Most time-consuming: 30-90 minutes**
- ✅ **Cell 6**: Visualize Training History
- ✅ **Cell 7**: Evaluate Both Models
- ✅ **Cell 8**: Ensemble Predictions
- ✅ **Cell 9**: Save Models
- ✅ **Cell 10**: Prediction Functions
- ✅ **Cell 11**: Example Predictions
- ✅ **Cell 12**: Summary Report

#### **What Augmentation Does During Training:**
```
Original Image → 
  Rotated (±25°) →
  Zoomed (80-120%) →
  Shifted (±20%) →
  Brightened/Darkened (80-120%) →
  Flipped Horizontally →
Augmented Image (fed to model)
```

Each epoch sees different augmented versions of the same images!

#### **Expected Output:**
```
Model Comparison:
  Custom CNN        → ~88-92% accuracy
  Transfer Learning → ~90-94% accuracy
  Ensemble (Hybrid) → ~92-95% accuracy ✓ Best!
```

#### **Download Results:**
All files saved to: `My Drive/Model_Outputs/`
- `model_custom_cnn_final.h5` - Custom CNN model
- `model_transfer_learning_final.h5` - Transfer learning model
- `training_history.png` - Loss/accuracy plots
- `confusion_matrices.png` - Prediction breakdown
- `example_predictions.png` - Sample predictions

---

### ⚡ GPU Acceleration Tips
- Colab provides free GPU (NVIDIA Tesla K80 or better)
- Check GPU: **Runtime** → **Change runtime type** → **Hardware accelerator: GPU**
- Verify: ✓ GPU available in Cell 1
- Training will be **10-50x faster** with GPU vs CPU!

### 🔥 Pro Tips
1. **Monitor Memory**: If notebook crashes, reduce `BATCH_SIZE` from 32 to 16
2. **Save Checkpoints**: Best models auto-saved during training
3. **Early Stopping**: Training stops if validation loss doesn't improve for 10 epochs
4. **Fine-tuning**: Unfreeze MobileNetV2 layers for better accuracy (advanced)

### 📖 Understanding the Hybrid Model
- **Custom CNN**: Learns banana disease patterns from scratch
- **Transfer Learning**: Uses ImageNet knowledge for generalization
- **Ensemble**: Combines both → `Prediction = (CNN + TL) / 2`
- **Result**: Better accuracy + robustness + confidence

---

✅ Ready to train! Start with **Cell 1** in the training section above.